# Market Features - Quick Excel Report

Este notebook genera un reporte rápido en Excel de las features de mercado calculadas.

**Inputs necesarios:**
- Candles (OHLCV)
- Orderbook snapshots
- Trades

**Output:**
- Excel file con todas las features calculadas por ventana de tiempo

In [1]:
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
import sys
import os

from research_notebooks.eda_strategies.rlmm.features.market_loader import MarketLoader

root_path = os.getcwd()
sys.path.append(root_path)

from src.market_features import compute_all_features
from src.config_loader import load_config

## 1. Configuración

Carga el archivo de configuración y ajusta el intervalo de resampling si es necesario.

In [2]:
# Cargar configuración
config_path = 'config/market_report.yml'

# Cargar configuración sin validación estricta (ya que simplificamos el YAML)
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Configuración cargada:")
print(f"  - Resampling interval: {config['resampling']['interval']}")
print(f"  - Label: {config['resampling']['label']}")
print(f"  - Include partial: {config['resampling']['include_partial']}")

Configuración cargada:
  - Resampling interval: 3h
  - Label: end
  - Include partial: False


### Ajustar intervalo de resampling (opcional)

Puedes cambiar el intervalo de resampling aquí:
- "1H" = 1 hora
- "30T" = 30 minutos
- "3H" = 3 horas
- "1D" = 1 día
- etc.

In [3]:
# Cambiar intervalo si es necesario
# config['resampling']['interval'] = '30T'  # Por ejemplo, 30 minutos

print(f"Intervalo final: {config['resampling']['interval']}")

Intervalo final: 3h


## 2. Cargar Datos

Carga tus datos de candles, orderbook y trades.

**Formato esperado:**

**Candles:**
- timestamp (datetime)
- open (float)
- high (float)
- low (float)
- close (float)
- volume (float)

**Orderbook:**
- timestamp (datetime)
- best_bid (float)
- best_ask (float)
- best_bid_size (float) - opcional
- best_ask_size (float) - opcional

**Trades:**
- timestamp (datetime)
- price (float)
- amount (float)
- side (str) - 'buy' o 'sell' (opcional, si no está se usa tick rule)

In [4]:
exchange = "binance"
trading_pair = "USDT-BRL"
date_range = ("2025-10-11", "2025-10-12")

candles_path = os.path.join(root_path, "data", "candles", f"{exchange}_{trading_pair}_1m_candles.csv")
orderbook_path = os.path.join(root_path, "data", "order_book")
trades_path = os.path.join(root_path, "data", "trades")
market_loader = MarketLoader(exchange=exchange, trading_pair=trading_pair, date_range=date_range)
candles, orderbook, trades = market_loader.load_all(candles_path, orderbook_path, trades_path)

print(f"\nDatos cargados:")
print(f"  - Candles: {len(candles)} filas")
print(f"  - Orderbook: {len(orderbook)} snapshots")
print(f"  - Trades: {len(trades)} trades")


Datos cargados:
  - Candles: 2880 filas
  - Orderbook: 172549 snapshots
  - Trades: 151733 trades


### Verificar datos

In [5]:
print("Candles:")
display(candles.head())
print(f"\nRango temporal: {candles['timestamp'].min()} a {candles['timestamp'].max()}")

print("\n" + "="*80)
print("Orderbook:")
display(orderbook.head())
print(f"\nRango temporal: {orderbook['timestamp'].min()} a {orderbook['timestamp'].max()}")

print("\n" + "="*80)
print("Trades:")
display(trades.head())
print(f"\nRango temporal: {trades['timestamp'].min()} a {trades['timestamp'].max()}")

Candles:


,timestamp,open,high,low,close,volume,quote_asset_volume,n_trades,taker_buy_base_volume,taker_buy_quote_volume
0,2025-10-11 00:00:00+00:00,5.6024,5.6050,5.6023,5.6040,74689.0,4.184667e+05,331.0,55444.6,310647.98532
1,2025-10-11 00:01:00+00:00,5.6040,5.6040,5.6027,5.6034,124333.4,6.966386e+05,251.0,68936.2,386248.74984
2,2025-10-11 00:02:00+00:00,5.6034,5.6086,5.6023,5.6052,136844.1,7.668739e+05,312.0,66005.8,369911.52495
3,2025-10-11 00:03:00+00:00,5.6059,5.6101,5.6013,5.6013,212689.4,1.191991e+06,260.0,72072.0,403844.87794
4,2025-10-11 00:04:00+00:00,5.6014,5.6040,5.6014,5.6021,38136.8,2.136488e+05,123.0,13730.7,76921.07912



Rango temporal: 2025-10-11 00:00:00+00:00 a 2025-10-12 23:59:00+00:00

Orderbook:


,timestamp,best_bid,best_bid_size,best_ask,best_ask_size
0,2025-10-11 00:00:00+00:00,5.6023,34941.7,5.6024,10721.8
1,2025-10-11 00:00:01+00:00,5.6023,34941.7,5.6024,10543.4
2,2025-10-11 00:00:02+00:00,5.6023,34909.8,5.6024,10489.9
3,2025-10-11 00:00:03+00:00,5.6023,34822.3,5.6024,10352.5
4,2025-10-11 00:00:04+00:00,5.6023,34632.3,5.6024,6190.2



Rango temporal: 2025-10-11 00:00:00+00:00 a 2025-10-12 23:59:58+00:00

Trades:


,timestamp,price,size,side
0,2025-10-11 00:00:00.027000+00:00,5.6024,178.4,buy
1,2025-10-11 00:00:01.317000+00:00,5.6024,53.5,buy
2,2025-10-11 00:00:01.558000+00:00,5.6023,31.9,sell
3,2025-10-11 00:00:02.379000+00:00,5.6024,126.2,buy
4,2025-10-11 00:00:02.491000+00:00,5.6024,11.2,buy



Rango temporal: 2025-10-11 00:00:00.027000+00:00 a 2025-10-12 23:59:57.596000+00:00


## 3. Calcular Features

Ejecuta el cálculo de features con resampling.

In [6]:
print("Calculando features...")

# Calcular features con resampling
features_df = compute_all_features(
    candles=candles,
    orderbook=orderbook,
    trades=trades,
    config=config,
    resample=True  # True para obtener ventanas de tiempo, False para un solo cálculo
)

print(f"\nFeatures calculadas: {len(features_df)} ventanas de tiempo")
print(f"Columnas: {len(features_df.columns)} features")
print(f"\nPrimeras filas:")
display(features_df.head())

Calculando features...

Features calculadas: 15 ventanas de tiempo
Columnas: 39 features

Primeras filas:


,price_last_mid_price,price_price_avg,price_price_std,price_price_diff_pct_t_1,price_price_velocity,price_high,price_low,price_delta_high_low,price_return_volatility,price_microtrend_slope,...,trades_sell_volume,trades_taker_aggressiveness_ratio,trades_vwap,trades_volume_imbalance,trades_microstructural_volume_signature,trades_volume_burst_score,trades_average_trade_size,trades_volume_time_distribution,trades_trade_flow_imbalance,trades_inter_arrival_time
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-10-11 03:00:00.027000+00:00,5.5868,5.585920,0.013378,0.001790,0.000120,5.6260,5.5621,0.0639,0.000639,0.000256,...,7902297.1,0.910597,5.585061,-0.046793,14.886778,0.043912,585.901913,0.296698,-0.314086,0.419101
2025-10-11 06:00:00.027000+00:00,5.5834,5.596859,0.005025,-0.001791,-0.000313,5.6001,5.5834,0.0167,0.000106,-0.000184,...,775638.4,1.436436,5.596090,0.179129,5.893214,-0.651064,293.537279,0.479058,0.837780,1.677346
2025-10-11 09:00:00.027000+00:00,5.5840,5.584829,0.000961,-0.008953,-0.000187,5.5868,5.5825,0.0043,0.000056,-0.000232,...,512425.1,1.000928,5.585076,0.000464,10.360701,0.424961,336.724302,0.465154,-0.521488,3.547676
2025-10-11 12:00:00.027000+00:00,5.6007,5.591269,0.006149,0.000000,0.000047,5.6007,5.5801,0.0206,0.000099,0.000042,...,650328.7,3.057073,5.592120,0.507034,8.145620,0.469038,330.796239,0.166553,0.988540,1.353992
2025-10-11 15:00:00.027000+00:00,5.6169,5.621224,0.011120,0.044528,-0.000507,5.6454,5.5803,0.0651,0.000352,-0.000924,...,5419207.5,1.002041,5.622186,0.001020,11.245237,-0.416992,642.817715,0.457306,0.022236,0.639688


### Estadísticas descriptivas

In [7]:
print("Estadísticas descriptivas:")
display(features_df.describe().T)

Estadísticas descriptivas:


,count,mean,std,min,25%,50%,75%,max
price_last_mid_price,15.0,5.612047e+00,2.306494e-02,5.581900e+00,5.585400e+00,5.620000e+00,5.630700e+00,5.645700e+00
price_price_avg,15.0,5.613743e+00,2.255692e-02,5.584829e+00,5.592817e+00,5.621224e+00,5.634453e+00,5.644826e+00
price_price_std,15.0,7.592468e-03,5.497871e-03,9.612895e-04,3.982947e-03,5.932174e-03,9.777681e-03,1.878823e-02
price_price_diff_pct_t_1,15.0,5.214547e-03,1.544420e-02,-8.953353e-03,0.000000e+00,0.000000e+00,1.784425e-03,4.452836e-02
price_price_velocity,15.0,-5.244444e-05,3.455832e-04,-8.733333e-04,-2.166667e-04,2.000000e-05,8.000000e-05,4.933333e-04
price_high,15.0,5.627547e+00,2.241695e-02,5.586800e+00,5.612400e+00,5.633400e+00,5.645650e+00,5.650000e+00
price_low,15.0,5.597780e+00,2.394208e-02,5.562100e+00,5.581100e+00,5.583400e+00,5.621200e+00,5.630100e+00
price_delta_high_low,15.0,2.976667e-02,2.024749e-02,4.300000e-03,1.660000e-02,2.060000e-02,4.450000e-02,6.510000e-02
price_return_volatility,15.0,2.064699e-04,1.444539e-04,5.600302e-05,1.139693e-04,1.718239e-04,2.400228e-04,6.388536e-04
price_microtrend_slope,15.0,1.535714e-05,3.516044e-04,-9.242857e-04,-9.357143e-05,3.321429e-05,1.760714e-04,5.667857e-04


## 4. Exportar a Excel

Genera el archivo Excel con todas las features.

In [8]:
# Definir nombre del archivo de salida
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

timestamp_str = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
output_file = output_dir / f'market_features_{timestamp_str}.xlsx'

print(f"Exportando a: {output_file}")

# Remover timezone del índice si existe (Excel no soporta timestamps con timezone)
features_df_export = features_df.copy()
if features_df_export.index.tz is not None:
    features_df_export.index = features_df_export.index.tz_localize(None)

# Diccionario de traducción de nombres de features a español
feature_names_spanish = {
    # Price features
    'price_last_mid_price': 'Close (Mid Price)',
    'price_price_avg': 'Price Avg',
    'price_price_std': 'Price Std',
    'price_price_diff_pct_t_1': 'Price Diff % T-1',
    'price_price_velocity': 'Price Velocity',
    'price_high': 'High',
    'price_low': 'Low',
    'price_delta_high_low': 'High-Low Delta',
    'price_return_volatility': 'Returns Volatility',
    'price_microtrend_slope': 'Pendiente Micro-tendencia',
    'price_short_vs_long_velocity_ratio': 'Ratio Velocidad Corto/Largo',
    
    # Order book features
    'order_book_mid_price': 'Precio Medio (Book)',
    'order_book_best_bid': 'Mejor Oferta Compra',
    'order_book_best_ask': 'Mejor Oferta Venta',
    'order_book_spread_abs': 'Spread Absoluto',
    'order_book_spread_rel': 'Spread Relativo',
    'order_book_depth_5': 'Profundidad 5 Niveles',
    'order_book_depth_10': 'Profundidad 10 Niveles',
    'order_book_depth_20': 'Profundidad 20 Niveles',
    'order_book_total_bid_liquidity': 'Liquidez Total Compra',
    'order_book_total_ask_liquidity': 'Liquidez Total Venta',
    'order_book_liquidity_imbalance': 'Desbalance de Liquidez',
    'order_book_market_pressure_index': 'Índice Presión Mercado',
    'order_book_queue_dynamics': 'Dinámica de Cola',
    'order_book_order_book_convexity': 'Convexidad del Book',
    'order_book_slippage_cost': 'Costo de Slippage (%)',
    'order_book_order_book_entropy': 'Entropía del Book',
    
    # Trades features
    'trades_total_volume': 'Volumen Total',
    'trades_buy_volume': 'Volumen Compra',
    'trades_sell_volume': 'Volumen Venta',
    'trades_taker_aggressiveness_ratio': 'Ratio Agresividad Taker',
    'trades_vwap': 'VWAP',
    'trades_volume_imbalance': 'Desbalance de Volumen',
    'trades_microstructural_volume_signature': 'Firma Microestructural',
    'trades_volume_burst_score': 'Score Burst Volumen',
    'trades_average_trade_size': 'Tamaño Promedio Trade',
    'trades_volume_time_distribution': 'Distribución Temporal Vol.',
    'trades_trade_flow_imbalance': 'Desbalance Flujo Trades',
    'trades_inter_arrival_time': 'Tiempo Inter-arribo (s)'
}

# Crear versión pivotada para impresión (Feature como filas, Timestamps como columnas)
features_df_pivot = features_df_export.T
features_df_pivot.index.name = 'Feature'

# Renombrar índices a español
features_df_pivot_spanish = features_df_pivot.copy()
features_df_pivot_spanish.index = [feature_names_spanish.get(idx, idx) for idx in features_df_pivot_spanish.index]

# Formatear columnas de timestamp para mejor legibilidad
features_df_pivot_spanish.columns = [col.strftime('%Y-%m-%d %H:%M') for col in features_df_pivot_spanish.columns]

# Exportar a Excel con formato
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Hoja 1: Features en formato original (técnico)
    features_df_export.to_excel(writer, sheet_name='Features', index=True)
    
    # Hoja 2: Features en formato pivotado para impresión (español)
    features_df_pivot_spanish.to_excel(writer, sheet_name='Reporte Impresión', index=True)
    
    # Hoja 3: Estadísticas descriptivas
    stats_df = features_df_export.describe().T
    stats_df.index = [feature_names_spanish.get(idx, idx) for idx in stats_df.index]
    stats_df.to_excel(writer, sheet_name='Estadísticas', index=True)
    
    # Hoja 4: Correlaciones (si hay suficientes datos)
    if len(features_df_export) > 10:
        corr_matrix = features_df_export.corr()
        # Renombrar índices y columnas a español
        corr_matrix.index = [feature_names_spanish.get(idx, idx) for idx in corr_matrix.index]
        corr_matrix.columns = [feature_names_spanish.get(col, col) for col in corr_matrix.columns]
        corr_matrix.to_excel(writer, sheet_name='Correlaciones', index=True)
    
    # Hoja 5: Configuración
    config_df = pd.DataFrame([
        {'Parámetro': 'Intervalo de Resampling', 'Valor': config['resampling']['interval']},
        {'Parámetro': 'Etiqueta', 'Valor': config['resampling']['label']},
        {'Parámetro': 'Incluir Parciales', 'Valor': config['resampling']['include_partial']},
        {'Parámetro': 'Ventana Corta (Precio)', 'Valor': config['price']['short_window']},
        {'Parámetro': 'Ventana Larga (Precio)', 'Valor': config['price']['long_window']},
        {'Parámetro': 'Tamaño Orden Estándar', 'Valor': config['order_book']['standard_order_size']},
        {'Parámetro': 'Ventana Volumen (Trades)', 'Valor': config['trades']['volume_window']},
        {'Parámetro': 'Ventana Flujo (Trades)', 'Valor': config['trades']['flow_window']},
    ])
    config_df.to_excel(writer, sheet_name='Configuración', index=False)

print(f"\n✓ Archivo Excel generado exitosamente: {output_file}")
print(f"\nContenido del archivo:")
print(f"  - Sheet 'Features': {len(features_df_export)} filas x {len(features_df_export.columns)} columnas (formato técnico)")
print(f"  - Sheet 'Reporte Impresión': {len(features_df_pivot_spanish)} features x {len(features_df_pivot_spanish.columns)} timestamps (formato español)")
print(f"  - Sheet 'Estadísticas': Estadísticas descriptivas en español")
if len(features_df_export) > 10:
    print(f"  - Sheet 'Correlaciones': Matriz de correlaciones en español")
print(f"  - Sheet 'Configuración': Configuración utilizada")

Exportando a: output/market_features_20251126_231403.xlsx

✓ Archivo Excel generado exitosamente: output/market_features_20251126_231403.xlsx

Contenido del archivo:
  - Sheet 'Features': 15 filas x 39 columnas (formato técnico)
  - Sheet 'Reporte Impresión': 39 features x 15 timestamps (formato español)
  - Sheet 'Estadísticas': Estadísticas descriptivas en español
  - Sheet 'Correlaciones': Matriz de correlaciones en español
  - Sheet 'Configuración': Configuración utilizada


## 5. Visualización rápida (opcional)

Graficar algunas features clave.

In [9]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# Crear PDF con visualizaciones
pdf_file = output_dir / f'market_features_report_{timestamp_str}.pdf'

print(f"Generando PDF: {pdf_file}")

with PdfPages(pdf_file) as pdf:
    # Página 1: Portada con información del reporte
    fig = plt.figure(figsize=(11, 8.5))
    fig.text(0.5, 0.7, 'Reporte de Features de Mercado', 
             ha='center', va='center', fontsize=24, weight='bold')
    fig.text(0.5, 0.6, f'Par: {trading_pair}', 
             ha='center', va='center', fontsize=16)
    fig.text(0.5, 0.55, f'Exchange: {exchange.upper()}', 
             ha='center', va='center', fontsize=16)
    fig.text(0.5, 0.5, f'Período: {date_range[0]} a {date_range[1]}', 
             ha='center', va='center', fontsize=16)
    fig.text(0.5, 0.4, f'Intervalo de Análisis: {config["resampling"]["interval"]}', 
             ha='center', va='center', fontsize=14)
    fig.text(0.5, 0.35, f'Total Features: {len(features_df.columns)}', 
             ha='center', va='center', fontsize=14)
    fig.text(0.5, 0.3, f'Ventanas de Tiempo: {len(features_df)}', 
             ha='center', va='center', fontsize=14)
    fig.text(0.5, 0.1, f'Generado: {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")}', 
             ha='center', va='center', fontsize=10, style='italic')
    plt.axis('off')
    pdf.savefig(fig, bbox_inches='tight')
    plt.close()
    
    # Página 2: Visualizaciones principales
    fig, axes = plt.subplots(3, 2, figsize=(11, 14))
    fig.suptitle('Análisis de Features de Mercado', fontsize=16, weight='bold', y=0.995)
    
    # Price features
    axes[0, 0].plot(features_df.index, features_df['price_last_mid_price'], 
                    label='Precio Medio', linewidth=2, color='#1f77b4')
    axes[0, 0].set_title('Evolución del Precio', fontsize=12, weight='bold')
    axes[0, 0].set_ylabel('Precio (BRL)', fontsize=10)
    axes[0, 0].legend(fontsize=9)
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].tick_params(axis='x', rotation=45, labelsize=8)
    
    axes[0, 1].plot(features_df.index, features_df['price_return_volatility'] * 100, 
                    label='Volatilidad', color='#d62728', linewidth=2)
    axes[0, 1].set_title('Volatilidad de Retornos', fontsize=12, weight='bold')
    axes[0, 1].set_ylabel('Volatilidad (%)', fontsize=10)
    axes[0, 1].legend(fontsize=9)
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].tick_params(axis='x', rotation=45, labelsize=8)
    
    # Order book features
    axes[1, 0].plot(features_df.index, features_df['order_book_spread_rel'] * 10000, 
                    label='Spread Relativo', color='#ff7f0e', linewidth=2)
    axes[1, 0].set_title('Spread Relativo (bps)', fontsize=12, weight='bold')
    axes[1, 0].set_ylabel('Spread (bps)', fontsize=10)
    axes[1, 0].legend(fontsize=9)
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].tick_params(axis='x', rotation=45, labelsize=8)
    
    axes[1, 1].plot(features_df.index, features_df['order_book_liquidity_imbalance'], 
                    label='Desbalance Liquidez', color='#9467bd', linewidth=2)
    axes[1, 1].axhline(y=0, color='black', linestyle='--', alpha=0.5, linewidth=1)
    axes[1, 1].set_title('Desbalance de Liquidez', fontsize=12, weight='bold')
    axes[1, 1].set_ylabel('Imbalance', fontsize=10)
    axes[1, 1].legend(fontsize=9)
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].tick_params(axis='x', rotation=45, labelsize=8)
    
    # Trades features
    axes[2, 0].plot(features_df.index, features_df['trades_total_volume'], 
                    label='Volumen Total', color='#2ca02c', linewidth=2)
    axes[2, 0].set_title('Volumen de Trading', fontsize=12, weight='bold')
    axes[2, 0].set_ylabel('Volumen', fontsize=10)
    axes[2, 0].legend(fontsize=9)
    axes[2, 0].grid(True, alpha=0.3)
    axes[2, 0].tick_params(axis='x', rotation=45, labelsize=8)
    
    axes[2, 1].plot(features_df.index, features_df['trades_volume_imbalance'], 
                    label='Desbalance Volumen', color='#8c564b', linewidth=2)
    axes[2, 1].axhline(y=0, color='black', linestyle='--', alpha=0.5, linewidth=1)
    axes[2, 1].set_title('Desbalance de Volumen (Compra/Venta)', fontsize=12, weight='bold')
    axes[2, 1].set_ylabel('Imbalance', fontsize=10)
    axes[2, 1].legend(fontsize=9)
    axes[2, 1].grid(True, alpha=0.3)
    axes[2, 1].tick_params(axis='x', rotation=45, labelsize=8)
    
    plt.tight_layout(rect=[0, 0, 1, 0.99])
    pdf.savefig(fig, bbox_inches='tight')
    plt.close()
    
    # Página 3: Estadísticas descriptivas (tabla)
    fig, ax = plt.subplots(figsize=(11, 8.5))
    ax.axis('off')
    
    # Seleccionar features clave para la tabla
    key_features = [
        'price_last_mid_price', 'price_return_volatility',
        'order_book_spread_rel', 'order_book_liquidity_imbalance',
        'trades_total_volume', 'trades_volume_imbalance'
    ]
    
    stats_summary = features_df[key_features].describe().T
    stats_summary.index = [feature_names_spanish.get(idx, idx) for idx in stats_summary.index]
    
    # Crear tabla
    table_data = []
    table_data.append(['Feature', 'Media', 'Std', 'Mín', 'Máx'])
    for idx, row in stats_summary.iterrows():
        table_data.append([
            idx,
            f"{row['mean']:.4f}",
            f"{row['std']:.4f}",
            f"{row['min']:.4f}",
            f"{row['max']:.4f}"
        ])
    
    table = ax.table(cellText=table_data, cellLoc='left', loc='center',
                     colWidths=[0.4, 0.15, 0.15, 0.15, 0.15])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)
    
    # Estilo de la tabla
    for i in range(len(table_data)):
        for j in range(5):
            cell = table[(i, j)]
            if i == 0:  # Header
                cell.set_facecolor('#4472C4')
                cell.set_text_props(weight='bold', color='white')
            else:
                cell.set_facecolor('#E7E6E6' if i % 2 == 0 else 'white')
    
    ax.set_title('Estadísticas Descriptivas - Features Principales', 
                 fontsize=16, weight='bold', pad=20)
    
    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

print(f"\n✓ PDF generado exitosamente: {pdf_file}")
print(f"\nContenido del PDF:")
print(f"  - Página 1: Portada con información del reporte")
print(f"  - Página 2: Visualizaciones principales (6 gráficos)")
print(f"  - Página 3: Tabla de estadísticas descriptivas")

print("\n✓ Visualización completada")

Generando PDF: output/market_features_report_20251126_231403.pdf

✓ PDF generado exitosamente: output/market_features_report_20251126_231403.pdf

Contenido del PDF:
  - Página 1: Portada con información del reporte
  - Página 2: Visualizaciones principales (6 gráficos)
  - Página 3: Tabla de estadísticas descriptivas

✓ Visualización completada


## 6. Resumen

Features calculadas por categoría:

In [10]:
# Separar columnas por categoría
price_cols = [col for col in features_df.columns if col.startswith('price_')]
order_book_cols = [col for col in features_df.columns if col.startswith('order_book_')]
trades_cols = [col for col in features_df.columns if col.startswith('trades_')]

print("=" * 80)
print("RESUMEN DE FEATURES CALCULADAS")
print("=" * 80)

print(f"\nPRICE FEATURES ({len(price_cols)}):")
for col in price_cols:
    print(f"  - {col}")

print(f"\nORDER BOOK FEATURES ({len(order_book_cols)}):")
for col in order_book_cols:
    print(f"  - {col}")

print(f"\nTRADES FEATURES ({len(trades_cols)}):")
for col in trades_cols:
    print(f"  - {col}")

print("\n" + "=" * 80)
print(f"TOTAL: {len(features_df.columns)} features calculadas")
print(f"VENTANAS DE TIEMPO: {len(features_df)}")
print(f"INTERVALO: {config['resampling']['interval']}")
print("=" * 80)

RESUMEN DE FEATURES CALCULADAS

PRICE FEATURES (11):
  - price_last_mid_price
  - price_price_avg
  - price_price_std
  - price_price_diff_pct_t_1
  - price_price_velocity
  - price_high
  - price_low
  - price_delta_high_low
  - price_return_volatility
  - price_microtrend_slope
  - price_short_vs_long_velocity_ratio

ORDER BOOK FEATURES (16):
  - order_book_mid_price
  - order_book_best_bid
  - order_book_best_ask
  - order_book_spread_abs
  - order_book_spread_rel
  - order_book_depth_5
  - order_book_depth_10
  - order_book_depth_20
  - order_book_total_bid_liquidity
  - order_book_total_ask_liquidity
  - order_book_liquidity_imbalance
  - order_book_market_pressure_index
  - order_book_queue_dynamics
  - order_book_order_book_convexity
  - order_book_slippage_cost
  - order_book_order_book_entropy

TRADES FEATURES (12):
  - trades_total_volume
  - trades_buy_volume
  - trades_sell_volume
  - trades_taker_aggressiveness_ratio
  - trades_vwap
  - trades_volume_imbalance
  - trades_m